In [0]:
%sql
DECLARE OR REPLACE VARIABLE v_report_month DATE DEFAULT DATE'2025-01-01';

DELETE FROM data_exploration_hub.marketing_hub.mkt_dashboard_tickets_monthly
WHERE report_month BETWEEN DATE'2025-01-01' AND DATE'2026-08-01';

BEGIN
WHILE v_report_month <= DATE'2026-08-01' DO

    INSERT INTO data_exploration_hub.marketing_hub.mkt_dashboard_tickets_monthly
    WITH month_bounds AS (
        SELECT
            v_report_month                                AS month_start,
            LAST_DAY(v_report_month)                       AS month_end,
            ADD_MONTHS(v_report_month, -1)                 AS prior_month_start,
            LAST_DAY(ADD_MONTHS(v_report_month, -1))       AS prior_month_end
    ),

    ticket_events AS (
        SELECT
            t.CDE_ID                            AS cde_id,
            d.datefull                          AS datefull,
            t.CineClubFLAG                      AS cineclub_flag,
            t.SCENEFLAG                         AS scene_flag,
            tk.paid_attendance_inflag           AS paid_flag,
            t.CineClub_Discount_Type_ID         AS discount_type_id,
            t.Quantity                          AS quantity,
            s.Auditorium_Experience_ID          AS auditorium_experience_id,
            s.Performance_Experience_ID         AS performance_experience_id,
            s.Is_DBOXflag                       AS is_dbox_flag
        FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc
            ON t.LocationId = loc.LocationID
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d
            ON t.VisitDateId = d.dateid
        LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_session s
            ON t.sessionskey = s.sessionskey
        LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_ticket_type tk
            ON t.TicketTypeSkey = tk.tickettypeskey
        CROSS JOIN month_bounds mb
        WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
          AND t.TicketFLAG = 'Yes'
          AND d.datefull BETWEEN mb.month_start AND mb.month_end
    ),

    ticket_events_classified AS (
        SELECT
            *,
            CASE
                WHEN auditorium_experience_id = 2      THEN 'AVX'
                WHEN auditorium_experience_id = 3      THEN 'IMAX'
                WHEN auditorium_experience_id = 4      THEN 'VIP'
                WHEN auditorium_experience_id IN (5,6) THEN 'ALT'
                WHEN auditorium_experience_id = 7      THEN 'FourDX'
                WHEN auditorium_experience_id = 8      THEN 'ScreenX'
                WHEN performance_experience_id = 2     THEN 'ThreeD'
                ELSE 'Regular'
            END AS final_format
        FROM ticket_events
    ),

    tickets_agg AS (
        SELECT
            'CC' AS population_type,
            COUNT(DISTINCT cde_id)                                                                     AS distinct_members,
            COUNT(DISTINCT cde_id, datefull)                                                           AS total_visits,
            SUM(CASE WHEN final_format = 'Regular' THEN quantity ELSE 0 END)                          AS tickets_regular,
            SUM(CASE WHEN final_format != 'Regular' THEN quantity ELSE 0 END)                          AS tickets_upgrades,
            SUM(CASE WHEN is_dbox_flag = 'Yes' THEN quantity ELSE 0 END)                               AS tickets_dbox,
            SUM(CASE WHEN final_format = 'AVX' THEN quantity ELSE 0 END)                               AS tickets_avx,
            SUM(CASE WHEN final_format = 'IMAX' THEN quantity ELSE 0 END)                              AS tickets_imax,
            SUM(CASE WHEN final_format = 'VIP' THEN quantity ELSE 0 END)                               AS tickets_vip,
            SUM(CASE WHEN final_format = 'ALT' THEN quantity ELSE 0 END)                               AS tickets_alt,
            SUM(CASE WHEN final_format = 'FourDX' THEN quantity ELSE 0 END)                            AS tickets_fourdx,
            SUM(CASE WHEN final_format = 'ScreenX' THEN quantity ELSE 0 END)                           AS tickets_screenx,
            SUM(CASE WHEN final_format = 'ThreeD' THEN quantity ELSE 0 END)                            AS tickets_threed,
            SUM(CASE WHEN paid_flag IS DISTINCT FROM 'Yes' THEN quantity ELSE 0 END)                   AS tickets_free,
            SUM(CASE WHEN paid_flag = 'Yes' THEN quantity ELSE 0 END)                                  AS tickets_paid,
            SUM(CASE WHEN discount_type_id = 6 THEN quantity ELSE 0 END)                               AS tickets_free_redeemed,
            SUM(CASE WHEN discount_type_id = 5 THEN quantity ELSE 0 END)                               AS companion_tickets_redeemed,
            SUM(CASE WHEN discount_type_id = 1 THEN quantity ELSE 0 END)                               AS tickets_undiscounted
        FROM ticket_events_classified
        WHERE cineclub_flag = 'Yes' AND cde_id IS NOT NULL

        UNION ALL

        SELECT
            'Scene' AS population_type,
            COUNT(DISTINCT cde_id),
            COUNT(DISTINCT cde_id, datefull),
            SUM(CASE WHEN final_format = 'Regular' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format != 'Regular' THEN quantity ELSE 0 END),
            SUM(CASE WHEN is_dbox_flag = 'Yes' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'AVX' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'IMAX' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'VIP' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'ALT' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'FourDX' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'ScreenX' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'ThreeD' THEN quantity ELSE 0 END),
            SUM(CASE WHEN paid_flag IS DISTINCT FROM 'Yes' THEN quantity ELSE 0 END),
            SUM(CASE WHEN paid_flag = 'Yes' THEN quantity ELSE 0 END),
            CAST(NULL AS BIGINT),
            CAST(NULL AS BIGINT),
            CAST(NULL AS BIGINT)
        FROM ticket_events_classified
        WHERE scene_flag = 'Yes' AND cineclub_flag IS DISTINCT FROM 'Yes' AND cde_id IS NOT NULL

        UNION ALL

        SELECT
            'Total_Biz' AS population_type,
            COUNT(DISTINCT cde_id),
            COUNT(DISTINCT cde_id, datefull),
            SUM(CASE WHEN final_format = 'Regular' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format != 'Regular' THEN quantity ELSE 0 END),
            SUM(CASE WHEN is_dbox_flag = 'Yes' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'AVX' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'IMAX' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'VIP' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'ALT' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'FourDX' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'ScreenX' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'ThreeD' THEN quantity ELSE 0 END),
            SUM(CASE WHEN paid_flag IS DISTINCT FROM 'Yes' THEN quantity ELSE 0 END),
            SUM(CASE WHEN paid_flag = 'Yes' THEN quantity ELSE 0 END),
            CAST(NULL AS BIGINT),
            CAST(NULL AS BIGINT),
            CAST(NULL AS BIGINT)
        FROM ticket_events_classified
    ),

    -- ----------------------------------------------------------------------------
    -- Credited -- CC-only, from the subscription events log (not the snapshot).
    -- tickets_credited = distinct (member, current_term_start_datetime) cycles
    -- where a payment_succeeded or subscription_renewed event fired in-month.
    -- Deduping on (member, cycle) avoids double-counting cycles where both event
    -- types fire for the same renewal. No plan_id filter needed -- validated to
    -- produce the same result with or without a plan split.
    -- tickets_in_market is unchanged -- still the snapshot's recognitions_left
    -- standing balance, since it's a balance, not a flow events can reconstruct.
    -- ----------------------------------------------------------------------------
    credit_cycles AS (
        SELECT DISTINCT
            cc_user_profile_id,
            current_term_start_datetime
        FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
        CROSS JOIN month_bounds mb
        WHERE event_type IN ('payment_succeeded', 'subscription_renewed')
          AND current_term_start_datetime >= mb.month_start
          AND current_term_start_datetime < ADD_MONTHS(mb.month_start, 1)
    ),

    snapshot_current_members AS (
        SELECT
            cc_user_profile_id,
            recognitions_left
        FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot sn
        CROSS JOIN month_bounds mb
        WHERE sn.snapshot_date_id = DATE_FORMAT(mb.month_end, 'yyyyMMdd')
    ),

    recognitions AS (
        SELECT
            (SELECT COUNT(*) FROM credit_cycles)      AS tickets_credited,
            SUM(cur.recognitions_left)                AS tickets_in_market
        FROM snapshot_current_members cur
    )

    SELECT
        mb.month_start AS report_month,
        ta.population_type,
        ta.distinct_members,
        ta.total_visits,
        ta.tickets_regular,
        ta.tickets_upgrades,
        ta.tickets_dbox,
        ta.tickets_avx,
        ta.tickets_imax,
        ta.tickets_vip,
        ta.tickets_alt,
        ta.tickets_fourdx,
        ta.tickets_screenx,
        ta.tickets_threed,
        ta.tickets_free,
        ta.tickets_paid,
        ta.tickets_free_redeemed,
        ta.companion_tickets_redeemed,
        ta.tickets_undiscounted,
        CASE WHEN ta.population_type = 'CC' THEN r.tickets_credited  ELSE NULL END AS tickets_credited,
        CASE WHEN ta.population_type = 'CC' THEN r.tickets_in_market ELSE NULL END AS tickets_in_market
    FROM tickets_agg ta
    CROSS JOIN month_bounds mb
    CROSS JOIN recognitions r
    ORDER BY ta.population_type;

    SET v_report_month = add_months(v_report_month, 1);
END WHILE;
END;